In [ ]:
import control as ct
import numpy as np
from models.systems import build_dc_plant, build_dc_electrical_plant
from utils.plotting import plot



In [ ]:
dcmotor_params =dict(
    source="https://www2.ece.ohio-state.edu/~passino/lab2_rotary_dynamics.pdf",
    Kt = 0.00767,
    v_dc = 10,
    Lm = 0.1e-3,
    Rm=2.6,
    J=3.0e-7,
    b=0
)

In [ ]:
tau_el = dcmotor_params['Lm'] / dcmotor_params['Rm']
tau_el

In [ ]:
dc_plant = build_dc_plant(dcmotor_params)
dc_plant

In [ ]:
times = np.arange(0, 0.1, 0.0001)   # Simulation time
X0 = [0., 0.]  # i , omega 
u0 = [1, 0]  # current command, load 

In [ ]:

t, y = ct.input_output_response(dc_plant, times, u0, X0)


In [ ]:
plot(t, [y[1]])

In [ ]:
plot(t, y[0])

In [ ]:
linear_plant = dc_plant.linearize([0,0], [0,0])
tf_dc_plant = ct.ss2tf(linear_plant)
tf_dc_plant

In [ ]:
# transfer function from voltage to omega
tf_u_i_motor = tf_dc_plant.returnScipySignalLTI()[1][0]
tf_u_i_motor = ct.tf(tf_u_i_motor.num, tf_u_i_motor.den)

In [ ]:
tf_u_i_motor

In [ ]:
_, y_red = ct.input_output_response(ct.ss2io(tf_u_i_motor), times, 1, 0)


In [ ]:
rl_data = ct.root_locus(tf_u_i_motor)

In [ ]:
ss_reduced = ct.balred( ct.tf2ss(tf_u_i_motor), 1)
ss_reduced

In [ ]:
tf_reduced =ct.ss2tf(ss_reduced)
tf_reduced.bandwidth() 

In [ ]:
ct.pzmap(tf_reduced)

In [ ]:
_, y_red = ct.input_output_response(ct.ss2io(ss_reduced), times, 1, 0)


In [ ]:
plot(
    t, 
    [y[1],y_red],
    xlabel = "Time",
    ylabel= " Omega",
    title="Step",
    trace_names=["Full", "Reduced"]
    )

In [ ]:
dc_elec_plant = build_dc_electrical_plant(dcmotor_params)

In [ ]:
tf_elec_dc = ct.ss2tf(dc_elec_plant.linearize(0,0))

In [ ]:
tf_elec_dc.bandwidth()

In [ ]:
times = np.arange(0, 0.002, 0.00001)   # Simulation time

t, y_ele = ct.input_output_response(dc_elec_plant, times, 1, 0)
plot(
    t, 
    [y_ele],
    xlabel = "Time",
    ylabel= " i",
    title="Step",
    )

In [ ]:
s = ct.tf('s')

In [ ]:
kp = 10
ki=1
tf_con = kp  + kp*ki /s
tf_con

In [ ]:
tf_cl = ct.feedback(tf_elec_dc, tf_con, -1)
tf_cl

In [ ]:
tf_cl.bandwidth()

In [ ]:
times = np.arange(0, 0.002, 0.00001)   # Simulation time

t, y_ele = ct.input_output_response(ct.tf2io(tf_cl), times, 1, 0)
plot(
    t, 
    [y_ele],
    xlabel = "Time",
    ylabel= " i",
    title="Step",
    )